# Part B — Customer Intelligence

This notebook builds RFM features, compares K-means choices, profiles four customer segments, and trains a leakage-aware lapse model. All features use data before 2025-09-01; September activity defines the future label.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.customer_intelligence import run_customer_intelligence
result = run_customer_intelligence()
print(result["metadata"])

{'cohort_rows': 1156, 'lapsed_customers': 443, 'lapse_rate': 0.3832, 'training_rows': 924, 'test_rows': 232, 'selected_model': 'logistic_regression', 'feature_cutoff': '2025-09-01', 'label_window_end': '2025-10-01', 'pca_explained_variance': [0.6343, 0.218], 'segment_counts': {'Promising': 436, 'Premium': 419, 'At Risk': 182, 'Loyal': 138}}


## 1. RFM segment profiles

Recency is days since the last completed order at 2025-10-01, frequency is distinct completed orders, and monetary is completed-order spend.

In [2]:
display(result["profile"])
display(result["segments"].head())
print("Segmented customers:", len(result["segments"]))

,segment_id,customers,avg_recency,avg_frequency,avg_monetary,value_score,segment_name
0,2,419,8.77,19.40,3410.02,2.75,Premium
1,1,138,24.12,6.77,7129.70,2.50,Loyal
2,0,436,28.86,3.77,1057.62,1.50,Promising
3,3,182,109.04,2.20,685.99,0.75,At Risk


,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,Promising,False
1,C0002,28,7,4644.90,1,Loyal,False
2,C0003,7,22,3644.85,2,Premium,True
3,C0004,0,5,369.00,0,Promising,False
4,C0005,147,2,324.00,3,At Risk,False


Segmented customers: 1175


## 2. K-means and PCA diagnostics

Inertia and silhouette are reported for K=2 through K=8. K=4 is retained for interpretable Premium, Loyal, Promising, and At Risk groups.

In [3]:
display(result["diagnostics"])
print("PCA explained variance:", result["metadata"]["pca_explained_variance"])

,k,inertia,silhouette
0,2,1820.7606,0.4650
1,3,1268.2346,0.4452
2,4,743.0711,0.5255
3,5,617.3666,0.4805
4,6,503.1706,0.4819
5,7,408.4605,0.4163
6,8,366.5035,0.4123


PCA explained variance: [0.6343, 0.218]


## 3. Leakage-aware lapse model

Preprocessing is fit only inside each training fold. Logistic regression and random forest are tuned with five-fold cross-validation and evaluated on a held-out stratified test set.

In [4]:
display(result["evaluation"])
print("Selected model:", result["metadata"]["selected_model"])
print("Feature cutoff:", result["metadata"]["feature_cutoff"])
print("Label window end:", result["metadata"]["label_window_end"])

,model,best_parameters,cv_f1,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,confusion_matrix
0,logistic_regression,"{""model__C"": 0.1}",0.7032,0.7026,0.5758,0.8539,0.6878,0.7830,"[[87, 56], [13, 76]]"
1,random_forest,"{""model__max_depth"": 4, ""model__min_samples_le...",0.7010,0.6897,0.5630,0.8539,0.6786,0.7871,"[[84, 59], [13, 76]]"


Selected model: logistic_regression
Feature cutoff: 2025-09-01
Label window end: 2025-10-01


## 4. Retention recommendations

This list ranks held-out Premium customers by predicted lapse probability. It is an illustrative targeting list; Premium is a post-hoc whole-window segment, not part of leakage-free model evaluation.

In [5]:
display(result["retention"])
print("Persisted model:", result["paths"]["model"].relative_to(ROOT))

,customer_id,lapsed,lapse_probability
137,C0146,0,0.403087
94,C0100,0,0.355138
249,C0260,0,0.348425
157,C0168,0,0.345946
133,C0142,0,0.328621
1014,C1047,0,0.317576
468,C0485,0,0.311953
760,C0786,0,0.307048
593,C0615,0,0.298019
130,C0139,0,0.250569


Persisted model: models/lapse_model.joblib
